# 03 - Multi-Trial Benchmarks (Champions & Classical Baselines)

This notebook consolidates the **Post-Evolution Benchmark Evaluation Pipeline** into a unified, reproducible workflow:
1. **Champion Extraction & Balance Check**: Selects top-performing candidate algorithms from the current SQLite database and exports `results/benchmark/champions.json`.
2. **Pre-Flight Diagnostic Audit**: Scans existing IOH traces in `results/ioh_traces/` and reports workload status.
3. **LLaMEA Champions Benchmark Execution**: Executes $N=20$ paired BBOB-instance trials per champion and reports clean objective gap ($\Delta y = f(x) - f^*$).
4. **Classical Baselines Benchmark Execution**: Executes the same $N=20$ instances for baseline optimizers (`CMA-ES`, `DE`, `PSO`).

The protocol allows a **fixed 100,000 objective evaluations per trial** at every dimension, for champions and baselines. Synthesis targets 2D, 10D and 50D with the same allowance. Champions use a timeout of **60 seconds per trial**.
**Normal execution:** no experiments are explicitly selected for rerun. The workflow discovers available conditions, reuses compatible completed results and executes missing or incomplete work. `force_rerun=False` keeps completed results cached.
**Storage:** historical data is preserved under results/archives/. The local active database and code/checkpoint directories were refreshed for the new study. A separate Jupyter checkout requires its own preparation.
Results recorded under a different or unknown budget are marked for rerun, even with `force_rerun=false`; they are not resumed or counted as current completed coverage. Run both champion and baseline evaluation sections before refreshing analysis and figures.

The current schedule uses **instances 2–21**, with seed `42 + instance_id`; synthesis still uses instance 1. Multiple-instance synthesis is not implemented. Reuse requires matching budget, instance/seed identities and provenance, not just a completed run count. Restart the kernel before loading the updated settings.

In [1]:
# Setup paths and services
# Ensure project root src/ is in sys.path
import os
import sys
from pathlib import Path

cwd = Path('.').resolve()
root_dir = cwd.parent if cwd.name == 'notebooks' else cwd
src_dir = root_dir / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import pandas as pd
from IPython.display import display, HTML

from shared.config import RESULTS_DIR
from benchmark.infra.di import BenchmarkProvider
from benchmark.application.trials.coordinator import BenchmarkCoordinator
from benchmark.application.select_champions import ChampionSelectionService
from benchmark.infra.storage import SQLiteSynthesisReadRepository
from shared.infra.lifecycle import WorkflowScope
from benchmark.domain.services.resolvers import ModelNames


CHAMPIONS_PATH = RESULTS_DIR / 'benchmark' / 'champions.json'

# The benchmark protocol is explicit; candidate conditions come from the database.

print('✅ Benchmark evaluation environment initialized.')
primary_target = 1e-8
secondary_target = 1e-2

# Prepare services once; infrastructure owns shutdown cleanup.
if "services" in globals():
    services.close()
services = WorkflowScope(BenchmarkProvider(project_root=root_dir), context={}, request_context={})
benchmark_coordinator = services.get(BenchmarkCoordinator)
champion_selection = services.get(ChampionSelectionService)
synthesis_reader = services.get(SQLiteSynthesisReadRepository)
model_names = services.get(ModelNames)



✅ Benchmark evaluation environment initialized.


## 1. Champion Selection & Experiment Balance Check
Query the current synthesis database, inspect run balance, and export problem-specific champions to `results/benchmark/champions.json`.

In [2]:
summary, total_completed = synthesis_reader.get_experiment_balance()

# Refresh the exported champion snapshot from the database currently loaded in this project.
champions_dict, df_champions = champion_selection.export_champions(CHAMPIONS_PATH)
print(f'✅ Exported {len(df_champions)} champions to {CHAMPIONS_PATH}')

if summary.empty:
    raise RuntimeError('No completed experiments found in database. Please run Notebook 02 first.')

print(f'== Total Completed Experiments: {total_completed} ==')
print(summary.to_string(index=False))

    # The snapshot above is refreshed before evaluation, so it always matches the active DB.


✅ Exported 960 champions to /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/benchmark/champions.json
== Total Completed Experiments: 960 ==
 problem_id  dim  noise_std     mode prompt_strategy                               llm_name  completed_count
          1    2       0.00 explicit        baseline qwen2.5-coder-14b-instruct-q4_k_m.gguf                1
          1    2       0.00 explicit        baseline qwen2.5-coder-32b-instruct-q4_k_m.gguf                1
          1    2       0.00 explicit          guided qwen2.5-coder-14b-instruct-q4_k_m.gguf                1
          1    2       0.00 explicit          guided qwen2.5-coder-32b-instruct-q4_k_m.gguf                1
          1    2       0.00 explicit        thinking qwen2.5-coder-14b-instruct-q4_k_m.gguf                1
          1    2       0.00 explicit        thinking qwen2.5-coder-32b-instruct-q4_k_m.gguf                1
          1    2       0.00 explicit   vectorization qwen2.5-coder-14b-instruct-q4_k_m.gguf    

## 2. Pre-Flight Diagnostic Audit (Champions & Baselines Workload)
Audit pending vs. completed evaluation runs across all experimental conditions.

In [3]:
# ── Embedded HTML Dashboard Helper (Notebook-Local UI) ─────────────────────────
def render_html_dashboard(
    df_audit: pd.DataFrame,
    title: str = "Benchmark Evaluation Pre-Flight Audit",
    subtitle: str = "Real-time status of empirical evaluation runs",
    group_column: str = "model",
) -> str:
    """Render responsive HTML pre-flight dashboard for Jupyter display."""
    if df_audit.empty:
        return "<div>No audit data available.</div>"

    grp_col = (
        group_column
        if group_column in df_audit.columns
        else ("solver" if "solver" in df_audit.columns else df_audit.columns[0])
    )

    get_clean_model_label = model_names.get_clean_model_label

    summary_rows = []
    for name, grp in df_audit.groupby(grp_col):
        clean_name = get_clean_model_label(str(name)) if grp_col == "model" else str(name).upper()
        total = len(grp)
        completed = len(grp[grp["status"] == "COMPLETED"])
        pending = len(grp[grp["status"] == "PENDING"])
        needs_rerun = len(grp[grp["status"] == "NEEDS_RERUN"])
        missing_code = len(grp[grp["status"] == "MISSING_CODE"])
        to_run_mask = grp["status"].isin(["PENDING", "NEEDS_RERUN"])
        if "is_filtered" in grp.columns:
            to_run_mask = to_run_mask & (~grp["is_filtered"])
        to_run = len(grp[to_run_mask])
        pct = (completed / total * 100) if total > 0 else 0.0
        summary_rows.append({
            "Group": clean_name,
            "Total Tasks": total,
            "Completed": completed,
            "Pending": pending,
            "Needs Rerun": needs_rerun,
            "Missing Code": missing_code,
            "Queue to Run": to_run,
            "Progress (%)": pct,
        })

    df_summary = pd.DataFrame(summary_rows)
    total_queue = int(df_summary["Queue to Run"].sum())
    total_completed = int(df_summary["Completed"].sum())
    total_tasks = int(df_summary["Total Tasks"].sum())
    overall_pct = (total_completed / total_tasks * 100) if total_tasks > 0 else 0.0

    html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 950px; margin: 15px 0;">
    <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 16px; border-bottom: 2px solid #E2E8F0; padding-bottom: 8px;">
        <div>
            <h2 style="margin: 0; color: #0F172A; font-size: 20px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
                🚀 {title}
            </h2>
            <p style="margin: 4px 0 0 0; color: #64748B; font-size: 13px;">{subtitle}</p>
        </div>
        <div style="background: #EEF2F6; padding: 6px 12px; border-radius: 20px; font-size: 12px; font-weight: 600; color: #334155;">
            {total_tasks} Target Conditions
        </div>
    </div>

    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 20px;">
        <div style="background: linear-gradient(135deg, #1E293B 0%, #0F172A 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #94A3B8;">Total Targets</div>
            <div style="font-size: 24px; font-weight: 700; color: #F8FAFC; margin-top: 4px;">{total_tasks}</div>
            <div style="font-size: 11px; color: #64748B; margin-top: 2px;">Across {len(df_summary)} Categories</div>
        </div>
        <div style="background: linear-gradient(135deg, #065F46 0%, #047857 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #A7F3D0;">Completed & Valid</div>
            <div style="font-size: 24px; font-weight: 700; color: #ECFDF5; margin-top: 4px;">{total_completed}</div>
            <div style="font-size: 11px; color: #D1FAE5; margin-top: 2px;">{overall_pct:.1f}% Overall Progress</div>
        </div>
        <div style="background: linear-gradient(135deg, #C2410C 0%, #9A3412 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #FED7AA;">Queue to Run</div>
            <div style="font-size: 24px; font-weight: 700; color: #FFF7ED; margin-top: 4px;">{total_queue}</div>
            <div style="font-size: 11px; color: #FFEDD5; margin-top: 2px;">Pending / Needs Rerun</div>
        </div>
        <div style="background: linear-gradient(135deg, #4338CA 0%, #3730A3 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #C7D2FE;">Overall Status</div>
            <div style="font-size: 24px; font-weight: 700; color: #EEF2FF; margin-top: 4px;">{overall_pct:.0f}%</div>
            <div style="font-size: 11px; color: #E0E7FF; margin-top: 2px;">Completion Rate</div>
        </div>
    </div>

    <div style="background: #F8FAFC; border: 1px solid #E2E8F0; border-radius: 10px; padding: 16px; margin-bottom: 8px;">
        <div style="font-size: 13px; font-weight: 700; color: #1E293B; margin-bottom: 12px; text-transform: uppercase; letter-spacing: 0.04em;">
            Completion Progress Breakdown
        </div>
"""
    for _, row in df_summary.iterrows():
        grp_name = row["Group"]
        tot = int(row["Total Tasks"])
        comp = int(row["Completed"])
        pend = int(row["Pending"])
        rerun = int(row["Needs Rerun"])
        q = int(row["Queue to Run"])
        pct = float(row["Progress (%)"])
        color = "#10B981" if pct > 75 else ("#3B82F6" if pct > 25 else "#F59E0B")
        rerun_pct = (rerun / tot * 100) if tot > 0 else 0

        html += f"""
        <div style="margin-bottom: 14px;">
            <div style="display: flex; justify-content: space-between; font-size: 13px; font-weight: 600; color: #334155; margin-bottom: 4px;">
                <span>⚙️ <strong style="color: #0F172A;">{grp_name}</strong> &nbsp;({comp}/{tot} Completed)</span>
                <span style="color: {color}; font-weight: 700;">{pct:.1f}%</span>
            </div>
            <div style="background: #E2E8F0; border-radius: 6px; height: 10px; overflow: hidden; display: flex;">
                <div style="background: #10B981; width: {pct}%; transition: width 0.3s;"></div>
                <div style="background: #EF4444; width: {rerun_pct}%;"></div>
            </div>
            <div style="display: flex; gap: 14px; font-size: 11px; color: #64748B; margin-top: 5px;">
                <span>✅ Completed: <strong style="color: #059669;">{comp}</strong></span>
                <span>⏳ Pending: <strong style="color: #D97706;">{pend}</strong></span>
                <span>⚠️ Needs Rerun: <strong style="color: #DC2626;">{rerun}</strong></span>
                <span>🎯 Queue: <strong style="color: #2563EB;">{q}</strong></span>
            </div>
        </div>
"""
    html += """
    </div>
</div>
"""
    return html

# 2. Audit champions workload (general overview by model)
df_champ_audit = benchmark_coordinator.audit_workload(
    solver_type='champions',
    trials_per_condition=20,
    random_seed=42,
    first_instance_id=2,
    budget=100_000,
    trial_timeout_seconds=60.0,
    observation_trace_points=64,
    force_rerun=False,
    classical_baselines=['cmaes', 'de', 'pso'],
    noise_robustness_enabled=True,
    cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
)
print(f"🏆 Champions Workload: {len(df_champ_audit)} conditions ({df_champ_audit["status"].value_counts().to_dict()})")
display(HTML(render_html_dashboard(df_champ_audit, title="🏆 LLaMEA Champions Evaluation Audit", subtitle="Empirical benchmark evaluation status across LLM-evolved algorithms", group_column="model")))

# 3. Discover unique conditions for classical baselines (general overview by solver)
df_base_audit = benchmark_coordinator.audit_workload(
    solver_type='baselines',
    trials_per_condition=20,
    random_seed=42,
    first_instance_id=2,
    budget=100_000,
    trial_timeout_seconds=60.0,
    observation_trace_points=64,
    force_rerun=False,
    classical_baselines=['cmaes', 'de', 'pso'],
    noise_robustness_enabled=True,
    cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
)
print(f"⚙️ Baselines Workload: {len(df_base_audit)} conditions ({df_base_audit["status"].value_counts().to_dict()})")
display(HTML(render_html_dashboard(df_base_audit, title="⚙️ Classical Baselines Evaluation Audit", subtitle="Empirical benchmark evaluation status across CMA-ES, DE, and PSO", group_column="solver")))


🏆 Champions Workload: 960 conditions ({'COMPLETED': 960})


⚙️ Baselines Workload: 180 conditions ({'COMPLETED': 180})


## 3. Execute LLaMEA Champions Benchmark (N=20 Independent Runs)
Evaluate evolved algorithms across target BBOB functions with full IOHprofiler `.dat` and `.json` logging.

In [4]:
results_champ_df = benchmark_coordinator.run_benchmarks(
    solver_type='champions',
    trials_per_condition=20,
    random_seed=42,
    first_instance_id=2,
    budget=100_000,
    trial_timeout_seconds=60.0,
    observation_trace_points=64,
    force_rerun=False,
    classical_baselines=['cmaes', 'de', 'pso'],
    noise_robustness_enabled=True,
    cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
)
print(f'✅ LLaMEA Champions evaluation complete: {len(results_champ_df)} conditions processed.')



🚀 STARTING BENCHMARKS FOR CHAMPIONS
   960 target conditions to evaluate

[1/960] 🏆 Champion: qwen2.5-coder-14b-instruct-q4_k_m.gguf
  Mode:        explicit   Strategy:    baseline       Dim:         2D     Noise level: σ=0       Problem:     f1 (Sphere (f1))
  📦 [CACHED] 20 runs found (Median Δy Error: 26.775). Skipping.

[2/960] 🏆 Champion: qwen2.5-coder-14b-instruct-q4_k_m.gguf
  Mode:        explicit   Strategy:    baseline       Dim:         2D     Noise level: σ=0.05    Problem:     f1 (Sphere (f1))
  📦 [CACHED] 20 runs found (Median Δy Error: 0.005). Skipping.

[3/960] 🏆 Champion: qwen2.5-coder-14b-instruct-q4_k_m.gguf
  Mode:        explicit   Strategy:    baseline       Dim:         2D     Noise level: σ=0.1     Problem:     f1 (Sphere (f1))
  📦 [CACHED] 20 runs found (Median Δy Error: 0.000). Skipping.

[4/960] 🏆 Champion: qwen2.5-coder-14b-instruct-q4_k_m.gguf
  Mode:        explicit   Strategy:    baseline       Dim:         2D     Noise level: σ=0.2     Problem:     f1 (S

## 4. Execute Classical Baselines Benchmark (CMA-ES, DE, PSO)
Run classical baseline optimizers across identical problem conditions for rigorous comparative benchmark.

In [5]:
results_base_df = benchmark_coordinator.run_benchmarks(
    solver_type='baselines',
    trials_per_condition=20,
    random_seed=42,
    first_instance_id=2,
    budget=100_000,
    trial_timeout_seconds=60.0,
    observation_trace_points=64,
    force_rerun=False,
    classical_baselines=['cmaes', 'de', 'pso'],
    noise_robustness_enabled=True,
    cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
)
print(f'✅ Classical Baselines evaluation complete: {len(results_base_df)} conditions processed.')



🚀 STARTING BENCHMARKS FOR BASELINES
   180 target conditions to evaluate

[1/180] ⚙️ Baseline: CMA-ES
  Mode:        —          Strategy:    —              Dim:         2D     Noise level: σ=0       Problem:     f1 (Sphere (f1))
  📦 [CACHED] 20 runs found (Median Δy Error: 0.000). Skipping.

[2/180] ⚙️ Baseline: CMA-ES
  Mode:        —          Strategy:    —              Dim:         2D     Noise level: σ=0       Problem:     f8 (Rosenbrock (f8))
  📦 [CACHED] 20 runs found (Median Δy Error: 0.000). Skipping.

[3/180] ⚙️ Baseline: CMA-ES
  Mode:        —          Strategy:    —              Dim:         2D     Noise level: σ=0       Problem:     f11 (Discus (f11))
  📦 [CACHED] 20 runs found (Median Δy Error: 0.000). Skipping.

[4/180] ⚙️ Baseline: CMA-ES
  Mode:        —          Strategy:    —              Dim:         2D     Noise level: σ=0       Problem:     f15 (Rastrigin Multi-Modal (f15))
  📦 [CACHED] 20 runs found (Median Δy Error: 1.990). Skipping.

[5/180] ⚙️ Baseline: CMA-E

## Separate robustness and generalization campaigns

Noise robustness freezes a clean champion and varies noise on its original function.
Cross-function generalization freezes a clean, explicit, baseline-strategy champion
and changes only the target function at the original dimension. These are secondary
analyses, not part of the native champion ranking. Trials use instances 2–21,
excluding synthesis instance 1. Previously inspected instances 2–20 are not a new untouched test set.
Set cross_function_enabled=True in the setup parameters before running it.
The switches below deliberately default to audit-only: no extra campaign runs on
notebook execution unless explicitly requested.


In [6]:
RUN_NOISE_ROBUSTNESS = False
RUN_CROSS_FUNCTION = False

noise_audit = benchmark_coordinator.audit_workload(
    solver_type='noise_robustness',
    trials_per_condition=20,
    random_seed=42,
    first_instance_id=2,
    budget=100_000,
    trial_timeout_seconds=60.0,
    observation_trace_points=64,
    force_rerun=False,
    classical_baselines=['cmaes', 'de', 'pso'],
    noise_robustness_enabled=True,
    cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
)
transfer_audit = benchmark_coordinator.audit_workload(
    solver_type='cross_function',
    trials_per_condition=20,
    random_seed=42,
    first_instance_id=2,
    budget=100_000,
    trial_timeout_seconds=60.0,
    observation_trace_points=64,
    force_rerun=False,
    classical_baselines=['cmaes', 'de', 'pso'],
    noise_robustness_enabled=True,
    cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
)
print(f"Noise robustness: {len(noise_audit)} conditions")
print(f"Cross-function generalization: {len(transfer_audit)} conditions (opt-in)")
display(noise_audit)
display(transfer_audit)

if RUN_NOISE_ROBUSTNESS:
    noise_results = benchmark_coordinator.run_benchmarks(
        solver_type='noise_robustness',
        trials_per_condition=20,
        random_seed=42,
        first_instance_id=2,
        budget=100_000,
        trial_timeout_seconds=60.0,
        observation_trace_points=64,
        force_rerun=False,
        classical_baselines=['cmaes', 'de', 'pso'],
        noise_robustness_enabled=True,
        cross_function_enabled=False,
        cross_function_problem_ids=[1, 8, 11, 15, 21],
    )
if RUN_CROSS_FUNCTION:
    if transfer_audit.empty:
        raise ValueError("Set cross_function_enabled=True in the cross-function audit and run calls first.")
    transfer_results = benchmark_coordinator.run_benchmarks(
        solver_type='cross_function',
        trials_per_condition=20,
        random_seed=42,
        first_instance_id=2,
        budget=100_000,
        trial_timeout_seconds=60.0,
        observation_trace_points=64,
        force_rerun=False,
        classical_baselines=['cmaes', 'de', 'pso'],
        noise_robustness_enabled=True,
        cross_function_enabled=False,
        cross_function_problem_ids=[1, 8, 11, 15, 21],
    )

# Always export the current readiness matrix, without treating missing trials as failures.
from shared.config import RESULTS_DIR
transfer_report_dir = RESULTS_DIR / "reports"
transfer_report_dir.mkdir(parents=True, exist_ok=True)
benchmark_coordinator.audit_workload(
    solver_type='noise_robustness',
    trials_per_condition=20,
    random_seed=42,
    first_instance_id=2,
    budget=100_000,
    trial_timeout_seconds=60.0,
    observation_trace_points=64,
    force_rerun=False,
    classical_baselines=['cmaes', 'de', 'pso'],
    noise_robustness_enabled=True,
    cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
).to_csv(
    transfer_report_dir / "noise_robustness_readiness_v1.csv", index=False
)
benchmark_coordinator.audit_workload(
    solver_type='cross_function',
    trials_per_condition=20,
    random_seed=42,
    first_instance_id=2,
    budget=100_000,
    trial_timeout_seconds=60.0,
    observation_trace_points=64,
    force_rerun=False,
    classical_baselines=['cmaes', 'de', 'pso'],
    noise_robustness_enabled=True,
    cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
).to_csv(
    transfer_report_dir / "cross_function_readiness_v1.csv", index=False
)


Noise robustness: 360 conditions
Cross-function generalization: 0 conditions (opt-in)


,key,solver_type,solver,display_name,model,strategy,problem_id,dim,noise_std,mode,...,recorded_trials,reason,status,median_error,is_filtered,raw_key,source_problem_id,source_noise_std,code_hash,target_dir
0,qwen2.5-coder-14b-instruct-q4_k_m.gguf/f1_2D_e...,noise_robustness,qwen_14b_baseline,qwen2.5-coder-14b-instruct-q4_k_m.gguf (noise ...,qwen2.5-coder-14b-instruct-q4_k_m.gguf,baseline,1,2,0.05,explicit,...,0,missing_provenance,PENDING,None,False,qwen2.5-coder-14b-instruct-q4_k_m.gguf/f1_2D_e...,1,0.0,023d0991a81660c32ce599ba1d17e7dab34e127ea46dee...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...
1,qwen2.5-coder-14b-instruct-q4_k_m.gguf/f1_2D_e...,noise_robustness,qwen_14b_baseline,qwen2.5-coder-14b-instruct-q4_k_m.gguf (noise ...,qwen2.5-coder-14b-instruct-q4_k_m.gguf,baseline,1,2,0.10,explicit,...,0,missing_provenance,PENDING,None,False,qwen2.5-coder-14b-instruct-q4_k_m.gguf/f1_2D_e...,1,0.0,023d0991a81660c32ce599ba1d17e7dab34e127ea46dee...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...
2,qwen2.5-coder-14b-instruct-q4_k_m.gguf/f1_2D_e...,noise_robustness,qwen_14b_baseline,qwen2.5-coder-14b-instruct-q4_k_m.gguf (noise ...,qwen2.5-coder-14b-instruct-q4_k_m.gguf,baseline,1,2,0.20,explicit,...,0,missing_provenance,PENDING,None,False,qwen2.5-coder-14b-instruct-q4_k_m.gguf/f1_2D_e...,1,0.0,023d0991a81660c32ce599ba1d17e7dab34e127ea46dee...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...
3,qwen2.5-coder-14b-instruct-q4_k_m.gguf/f1_2D_e...,noise_robustness,qwen_14b_guided,qwen2.5-coder-14b-instruct-q4_k_m.gguf (noise ...,qwen2.5-coder-14b-instruct-q4_k_m.gguf,guided,1,2,0.05,explicit,...,0,missing_provenance,PENDING,None,False,qwen2.5-coder-14b-instruct-q4_k_m.gguf/f1_2D_e...,1,0.0,f7ec7551733aaf9ded2201aad1320b64df6940e82a672b...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...
4,qwen2.5-coder-14b-instruct-q4_k_m.gguf/f1_2D_e...,noise_robustness,qwen_14b_guided,qwen2.5-coder-14b-instruct-q4_k_m.gguf (noise ...,qwen2.5-coder-14b-instruct-q4_k_m.gguf,guided,1,2,0.10,explicit,...,0,missing_provenance,PENDING,None,False,qwen2.5-coder-14b-instruct-q4_k_m.gguf/f1_2D_e...,1,0.0,f7ec7551733aaf9ded2201aad1320b64df6940e82a672b...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_50D...,noise_robustness,qwen_32b_thinking,qwen2.5-coder-32b-instruct-q4_k_m.gguf (noise ...,qwen2.5-coder-32b-instruct-q4_k_m.gguf,thinking,21,50,0.10,explicit,...,0,missing_provenance,PENDING,None,False,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_50D...,21,0.0,3b32f93081c85d695a8437982f434f277a3cbdddb4a8dc...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...
356,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_50D...,noise_robustness,qwen_32b_thinking,qwen2.5-coder-32b-instruct-q4_k_m.gguf (noise ...,qwen2.5-coder-32b-instruct-q4_k_m.gguf,thinking,21,50,0.20,explicit,...,0,missing_provenance,PENDING,None,False,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_50D...,21,0.0,3b32f93081c85d695a8437982f434f277a3cbdddb4a8dc...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...
357,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_50D...,noise_robustness,qwen_32b_vectorization,qwen2.5-coder-32b-instruct-q4_k_m.gguf (noise ...,qwen2.5-coder-32b-instruct-q4_k_m.gguf,vectorization,21,50,0.05,explicit,...,0,missing_provenance,PENDING,None,False,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_50D...,21,0.0,24a96a0b714823c6c20fd7de7a7e08c29c73119f18ff5a...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...
358,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_50D...,noise_robustness,qwen_32b_vectorization,qwen2.5-coder-32b-instruct-q4_k_m.gguf (noise ...,qwen2.5-coder-32b-instruct-q4_k_m.gguf,vectorization,21,50,0.10,explicit,...,0,missing_provenance,PENDING,None,False,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_50D...,21,0.0,24a96a0b714823c6c20fd7de7a7e08c29c73119f18ff5a...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...


""


## 5. Post-evaluation outcome summary and synthesis review

Read-only summary of current native champions and baselines. This does not run experiments, change results or write reports. Historical unexecuted tails are excluded. Valid trials exclude execution failures; execution and failure rates use executed trials as their denominator. Target success rate counts trials reaching the primary error target, including failures in its denominator. The overall median includes failures, while the median finite error describes only successful executions. Missing/stale and incomplete conditions require benchmark completion before synthesis decisions. `all_failed` and `mostly_failed` (>50% execution failures) are review candidates, not automatic reruns. Median error above the configured secondary target is a performance flag, not proof of a bug. The full tables remain available as `evaluation_summary`, `failure_review` and `large_error_review`.


In [7]:
evaluation_summary = benchmark_coordinator.summarize_results(
    trials_per_condition=20,
    random_seed=42,
    first_instance_id=2,
    budget=100_000,
    trial_timeout_seconds=60.0,
    observation_trace_points=64,
    force_rerun=False,
    classical_baselines=['cmaes', 'de', 'pso'],
    noise_robustness_enabled=True,
    cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
    primary_target=primary_target,
    secondary_target=secondary_target,
)
if evaluation_summary.empty:
    print("No current benchmark conditions to summarize.")
else:
    print(f"Summary of {len(evaluation_summary)} conditions; primary target={primary_target:g}, secondary target={secondary_target:g}")
    display(pd.crosstab(evaluation_summary["Model"], evaluation_summary["Category"]))
    review_columns = ["Model", "Experiment ID", "Algorithm", "Mode", "Strategy", "Problem ID", "Dim", "Noise", "Executed trials", "Valid trials", "Expected trials", "Failed trials", "Execution rate", "Failure rate", "Solved trials", "Primary target rate", "Median error", "Median finite error", "Best baseline median error", "Category", "Suggested action"]
    failure_review = evaluation_summary[evaluation_summary["Category"].isin(["all_failed", "mostly_failed", "some_failed", "incomplete", "missing_or_stale"])][review_columns]
    large_error_review = evaluation_summary[evaluation_summary["Category"] == "large_error"][review_columns]
    formats = {"Execution rate": "{:.1%}", "Failure rate": "{:.1%}", "Primary target rate": "{:.1%}", "Noise": "{:g}", "Median error": "{:.3f}", "Median finite error": "{:.3f}", "Best baseline median error": "{:.3f}"}
    print(f"Failure/readiness review: {len(failure_review)} conditions (first 100 shown).")
    display(failure_review.head(100).style.format(formats, na_rep="—"))
    print(f"Large finite-error review: {len(large_error_review)} conditions (first 50 shown; compare with baselines before rerunning).")
    display(large_error_review.head(50).style.format(formats, na_rep="—"))
    print("Suggested experiment IDs to REVIEW for fresh synthesis, grouped by owning LLM:")
    for model, group in evaluation_summary[evaluation_summary["Rerun candidate"]].groupby("Model", sort=True):
        print(f"{model}: {sorted(group['Experiment ID'].dropna().astype(int).unique().tolist())}")
    print("No reruns launched. Review diagnostics first; explicitly select IDs in Notebook 02/configuration.")


Summary of 1140 conditions; primary target=1e-08, secondary target=0.01


Category,all_failed,finite_results,large_error,mostly_failed,some_failed
Model,,,,,
cmaes,0,22,38,0,0
de,0,14,46,0,0
pso,0,19,41,0,0
qwen2.5-coder-14b-instruct-q4_k_m.gguf,1,130,337,5,7
qwen2.5-coder-32b-instruct-q4_k_m.gguf,0,175,298,3,4


Failure/readiness review: 20 conditions (first 100 shown).


,Model,Experiment ID,Algorithm,Mode,Strategy,Problem ID,Dim,Noise,Executed trials,Valid trials,Expected trials,Failed trials,Execution rate,Failure rate,Solved trials,Primary target rate,Median error,Median finite error,Best baseline median error,Category,Suggested action
218,qwen2.5-coder-14b-instruct-q4_k_m.gguf,5533,ImprovedCMAESOptimizer,explicit,vectorization,1,10,0,20,8,20,12,40.0%,60.0%,0,0.0%,inf,27.952,0.000,mostly_failed,Inspect; consider fresh synthesis
231,qwen2.5-coder-14b-instruct-q4_k_m.gguf,5554,EvolutionaryStrategy,implicit,guided,1,10,0.1,20,10,20,10,50.0%,50.0%,0,0.0%,inf,1.316,0.000,some_failed,Inspect failed-trial diagnostics
292,qwen2.5-coder-14b-instruct-q4_k_m.gguf,5579,NoisyCMAES,explicit,baseline,8,2,0.1,20,9,20,11,45.0%,55.0%,0,0.0%,inf,675.141,0.000,mostly_failed,Inspect; consider fresh synthesis
348,qwen2.5-coder-14b-instruct-q4_k_m.gguf,5379,EnhancedStochasticES,explicit,baseline,8,50,0.05,20,8,20,12,40.0%,60.0%,0,0.0%,inf,74.586,53.155,mostly_failed,Inspect; consider fresh synthesis
397,qwen2.5-coder-14b-instruct-q4_k_m.gguf,5655,GeneticAlgorithm,implicit,baseline,11,2,0.2,20,17,20,3,85.0%,15.0%,0,0.0%,69159382.828,65272112.605,0.000,some_failed,Inspect failed-trial diagnostics
417,qwen2.5-coder-14b-instruct-q4_k_m.gguf,5672,EnhancedGeneticAlgorithm,implicit,thinking,11,10,0.05,20,9,20,11,45.0%,55.0%,0,0.0%,inf,9.738,0.241,mostly_failed,Inspect; consider fresh synthesis
448,qwen2.5-coder-14b-instruct-q4_k_m.gguf,5412,EnhancedPSO,explicit,thinking,11,50,0.05,20,16,20,4,80.0%,20.0%,0,0.0%,387.459,358.564,151.004,some_failed,Inspect failed-trial diagnostics
454,qwen2.5-coder-14b-instruct-q4_k_m.gguf,5422,ImprovedStochasticOptimizationDE,explicit,guided,11,50,0.1,20,12,20,8,60.0%,40.0%,0,0.0%,851.044,651.437,323.486,some_failed,Inspect failed-trial diagnostics
500,qwen2.5-coder-14b-instruct-q4_k_m.gguf,5723,EnhancedGeneticAlgorithm,explicit,baseline,15,10,0,20,11,20,9,55.0%,45.0%,0,0.0%,22.925,10.124,11.442,some_failed,Inspect failed-trial diagnostics
565,qwen2.5-coder-14b-instruct-q4_k_m.gguf,5759,RefinedNoisyCMAES,implicit,baseline,21,2,0,20,18,20,2,90.0%,10.0%,0,0.0%,2.156,1.991,0.000,some_failed,Inspect failed-trial diagnostics


Large finite-error review: 760 conditions (first 50 shown; compare with baselines before rerunning).


,Model,Experiment ID,Algorithm,Mode,Strategy,Problem ID,Dim,Noise,Executed trials,Valid trials,Expected trials,Failed trials,Execution rate,Failure rate,Solved trials,Primary target rate,Median error,Median finite error,Best baseline median error,Category,Suggested action
15,cmaes,—,cmaes,explicit,classical,8,2,0.2,20,20,20,0,100.0%,0.0%,7,35.0%,0.222,0.222,0.000,large_error,Compare with baselines; not an automatic rerun
17,cmaes,—,cmaes,explicit,classical,8,10,0.05,20,20,20,0,100.0%,0.0%,0,0.0%,8.782,8.782,8.243,large_error,Compare with baselines; not an automatic rerun
18,cmaes,—,cmaes,explicit,classical,8,10,0.1,20,20,20,0,100.0%,0.0%,0,0.0%,9.507,9.507,8.378,large_error,Compare with baselines; not an automatic rerun
19,cmaes,—,cmaes,explicit,classical,8,10,0.2,20,20,20,0,100.0%,0.0%,0,0.0%,9.971,9.971,9.971,large_error,Compare with baselines; not an automatic rerun
21,cmaes,—,cmaes,explicit,classical,8,50,0.05,20,20,20,0,100.0%,0.0%,0,0.0%,53.155,53.155,53.155,large_error,Compare with baselines; not an automatic rerun
22,cmaes,—,cmaes,explicit,classical,8,50,0.1,20,20,20,0,100.0%,0.0%,0,0.0%,60.596,60.596,60.596,large_error,Compare with baselines; not an automatic rerun
23,cmaes,—,cmaes,explicit,classical,8,50,0.2,20,20,20,0,100.0%,0.0%,0,0.0%,219.353,219.353,219.353,large_error,Compare with baselines; not an automatic rerun
26,cmaes,—,cmaes,explicit,classical,11,2,0.1,20,20,20,0,100.0%,0.0%,10,50.0%,0.265,0.265,0.007,large_error,Compare with baselines; not an automatic rerun
29,cmaes,—,cmaes,explicit,classical,11,10,0.05,20,20,20,0,100.0%,0.0%,9,45.0%,9.065,9.065,0.241,large_error,Compare with baselines; not an automatic rerun
30,cmaes,—,cmaes,explicit,classical,11,10,0.1,20,20,20,0,100.0%,0.0%,8,40.0%,48.522,48.522,0.299,large_error,Compare with baselines; not an automatic rerun


Suggested experiment IDs to REVIEW for fresh synthesis, grouped by owning LLM:
qwen2.5-coder-14b-instruct-q4_k_m.gguf: [5379, 5479, 5494, 5533, 5579, 5672]
qwen2.5-coder-32b-instruct-q4_k_m.gguf: [5857, 5865, 5913]
No reruns launched. Review diagnostics first; explicitly select IDs in Notebook 02/configuration.
